# 🧬 Nghiên Cứu Protein: Khai Phá PubMed, Cấu Trúc 3D & Phân Tích Cơ Chế / Tác Dụng Phụ

Notebook này hướng dẫn quy trình tự động hóa nghiên cứu protein gồm:
1. **UniProt & PDB**: Lấy thông tin chức năng, gene, và cấu trúc 3D.
2. **3D Visualization**: Hiển thị mô hình phân tử 3D tương tác (`py3Dmol`) xoay 360° ngay trong Notebook.
3. **PubMed Mining**: Kéo các bài báo khoa học mới nhất từ NCBI PubMed E-Utilities.
4. **Phân Tích Cơ Chế Hoạt Động (Mechanism of Action)**: Trực quan hóa các cơ chế phân tử hàng đầu.
5. **Phân Tích Hiệu Ứng Phụ (Side Effects & Toxicity)**: Thống kê tần suất tác dụng phụ và trích xuất bằng chứng khoa học có dẫn chứng PMID.

In [ ]:
# 1. Import các thư viện cần thiết
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from pubmed_client import PubMedClient
from protein_client import ProteinClient
from analyzer import ProteinLiteratureAnalyzer
import visualizer

print("✓ Đã sẵn sàng các module nghiên cứu sinh học phân tử!")

## Bước 1: Tra cứu thông tin Protein trên UniProt và PDB
Bạn có thể thay đổi tên protein bất kỳ (ví dụ: `EGFR`, `TP53`, `KRAS`, `ACE2`, `HER2`, `BRAF`...)

In [ ]:
TARGET_PROTEIN = "EGFR"  # Epidermal Growth Factor Receptor

prot_client = ProteinClient()
protein_data = prot_client.search_uniprot(TARGET_PROTEIN)

if protein_data:
    print(f"Tên protein: {protein_data['protein_name']}")
    print(f"UniProt ID:  {protein_data['accession']}")
    print(f"Sinh vật:    {protein_data['organism']}")
    print(f"PDB tốt nhất:{protein_data['best_pdb_id']}")
    print("\n--- Tóm tắt chức năng sinh học ---")
    print(protein_data['function'][:450] + "...")
else:
    print(f"Không tìm thấy dữ liệu cho {TARGET_PROTEIN}")

## Bước 2: Hiển thị hình dáng Protein 3D tương tác (`py3Dmol`)
Bạn có thể dùng chuột kéo để xoay, lăn chuột để zoom.

In [ ]:
pdb_id = protein_data.get('best_pdb_id', '1IVO') if protein_data else '1IVO'
print(f"Hiển thị cấu trúc 3D của PDB ID: {pdb_id}")

# Hiển thị dạng dải xoắn ribbon/cartoon tô màu theo phổ quang phổ spectrum
viewer = visualizer.view_3d_protein(pdb_id=pdb_id, style="cartoon", color_scheme="spectrum", width=750, height=480)
viewer.show()

## Bước 3: Kéo các bài báo nghiên cứu từ PubMed
Sử dụng NCBI E-utilities để lấy bài báo, trích xuất Title, Abstract, Journal, Năm xuất bản, Tác giả và DOI.

In [ ]:
pubmed = PubMedClient()
MAX_ARTICLES = 30

print(f"Đang kéo {MAX_ARTICLES} bài báo mới nhất về {TARGET_PROTEIN} từ PubMed...")
df_articles = pubmed.search_protein(TARGET_PROTEIN, max_results=MAX_ARTICLES)
print(f"✓ Đã lấy về {len(df_articles)} bài báo.")

# Hiển thị 5 bài báo đầu tiên
df_articles[["pmid", "year", "title", "journal"]].head()

## Bước 4: Phân tích & Trực quan hóa Cách thức hoạt động (Mechanisms)

In [ ]:
analyzer = ProteinLiteratureAnalyzer()
findings = analyzer.summarize_findings(df_articles)

# Vẽ biểu đồ Top cơ chế hoạt động phân tử
fig, ax = plt.subplots(figsize=(9, 4.5))
visualizer.plot_mechanisms_chart(findings["top_mechanisms"], TARGET_PROTEIN, ax=ax)
plt.show()

## Bước 5: Phân tích & Trực quan hóa Hiệu ứng phụ / Độc tính (Side Effects)

In [ ]:
# Vẽ biểu đồ Top hiệu ứng phụ xuất hiện trong tài liệu y văn
fig, ax = plt.subplots(figsize=(9, 4.5))
visualizer.plot_side_effects_chart(findings["top_side_effects"], TARGET_PROTEIN, ax=ax)
plt.show()

## Bước 6: Trích xuất các câu bằng chứng khoa học (Evidence Sentences) kèm PMID

In [ ]:
print("=== BẰNG CHỨNG VỀ TÁC DỤNG PHỤ / ĐỘC TÍNH PHÁT HIỆN TRONG ABSTRACTS ===\n")
for i, ev in enumerate(findings["side_effect_evidence"], 1):
    print(f"[{i}] PMID: {ev['pmid']} ({ev['year']})")
    print(f"    Bài báo: {ev['title']}")
    print(f"    Trích dẫn: \"{ev['sentence']}\"")
    print(f"    Link: https://pubmed.ncbi.nlm.nih.gov/{ev['pmid']}/\n")

## Bước 7: Xuất toàn bộ báo cáo hoàn chỉnh (HTML & Markdown)

In [ ]:
from protein_report import generate_full_report

md_path, html_path = generate_full_report(TARGET_PROTEIN, max_articles=25, output_dir="./reports")
print(f"\n✓ Báo cáo hoàn chỉnh đã sẵn sàng tại: {html_path}")